# Module 15 — The Feed-Forward Block

Attention (Modules 10-11) is the part of a transformer where tokens
exchange information with each other — every position gathers context from
every earlier position. The **feed-forward block** is the opposite: a
small per-token MLP applied identically and *completely independently* to
each position, with zero mixing across tokens. If attention is "look
around and gather context," the feed-forward block is "now think about
what you gathered, on your own."

Architecture (same as GPT-2 and this project's Module 17 nanoGPT):
`Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)`, where `d_ff` is
typically **4x** `d_model`.

## 1. The block itself

In [ ]:
import torch
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
d_model = 16
ffn = FeedForward(d_model)
n_params = sum(p.numel() for p in ffn.parameters())
print(ffn)
print(f"\nd_ff = {4 * d_model} (4x d_model, the standard GPT-2-style expansion ratio)")
print(f"parameters: {n_params}")

## 2. Why GELU instead of ReLU

ReLU is `max(0, x)` — a hard cutoff with zero gradient for any negative
input (a "dead" neuron there learns nothing). GELU is a smooth
approximation of the same basic shape that stays (slightly) differentiable
everywhere, including near 0. It's a small change that empirically trains
better for transformers, which is why GPT-2 onward all use it instead of
plain ReLU.

In [ ]:
import matplotlib.pyplot as plt

xs = torch.linspace(-4, 4, 200)
relu_ys = torch.relu(xs)
gelu_ys = torch.nn.functional.gelu(xs)

plt.figure(figsize=(6, 4))
plt.plot(xs, relu_ys, label="ReLU")
plt.plot(xs, gelu_ys, label="GELU")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.legend()
plt.title("ReLU vs. GELU")
plt.show()

## 3. The key property: per-token, zero cross-token mixing

Unlike attention — which we showed in Module 12 mixes information across
positions (and is only permutation-*equivariant*, not fully independent,
even without positional encoding) — the feed-forward block processes each
token as if it were the only one in the sequence. Shuffle the tokens any
way you like, run the FFN, un-shuffle: you get back *exactly* the original
output, for any input, any weights, no exceptions. That's a much stronger
guarantee than what Module 12 proved for attention, and it's worth
confirming directly.

In [ ]:
seq_len = 6
x = torch.randn(seq_len, d_model)

out_original = ffn(x)

perm = torch.randperm(seq_len)
out_permuted = ffn(x[perm])

unshuffled = torch.zeros_like(out_permuted)
unshuffled[perm] = out_permuted

assert torch.allclose(unshuffled, out_original, atol=1e-6)
# even stronger: each row of the output depends ONLY on the same row of the input
for i in range(seq_len):
    single_token_out = ffn(x[i:i+1])
    assert torch.allclose(single_token_out[0], out_original[i], atol=1e-6)

print("Confirmed: every output row depends only on its own input row - true per-token independence, no cross-token mixing at all.")

## Recap

- The feed-forward block is `Linear → GELU → Linear`, expanding to `4x
  d_model` in the middle and back down — a per-token MLP.
- GELU is a smooth alternative to ReLU that transformers use in practice.
- It has a much stronger independence property than attention: each
  token's output depends **only** on that token's own input, verified
  directly (not just permutation-equivariant like attention — fully
  independent per position).

With attention (mixing across tokens) and the feed-forward block
(processing each token independently) both built, plus positional encoding,
layer norm, and residual connections, Module 16 assembles all five pieces
into one real transformer block.